# Crime Analysis Pipeline

In [69]:
# Import modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## Ingestion

### Population Data

### Deprivation Data

### Crime Severity Data

In [48]:
# Import crime severity categorised data set
sev = pd.read_csv(f'../Data/Processed/crime-severity-categorised.csv')

sev.head()

,Crime Index,Offence,Weight,Crime Category
0,"1, 4.1/10/2",Homicide,"7,979",Violence and sexual offences
1,2,Attempted murder,"4,663",Violence and sexual offences
2,4.3,Intentional destruction of viable unborn child,15,Violence and sexual offences
3,4.4,Causing death or serious injury by dangerous d...,"1,092",Violence and sexual offences
4,4.6,Causing death by careless driving when under t...,"1,595",Violence and sexual offences


### Crime Data

In [ ]:
## Import one months worth of data
police_region = 'merseyside'
year_month = '2026-03' # Get the most recent data available

mssd = pd.read_csv(f'../Data/Raw/{police_region}/{year_month}-{police_region}-street.csv')

mssd.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,NaN,2026-03,Merseyside Police,Merseyside Police,-2.871827,53.489763,On or near Gilescroft Avenue,E01006448,Knowsley 001A,Anti-social behaviour,NaN,NaN
1,ca09c84a698c13c40d825f23eefd27b3ce078c6ef79828...,2026-03,Merseyside Police,Merseyside Police,-2.874541,53.485420,On or near Harleston Road,E01006448,Knowsley 001A,Criminal damage and arson,Unable to prosecute suspect,NaN
2,72c3ba95abd85eb01d07a5443bb313dd6584a19a2052bd...,2026-03,Merseyside Police,Merseyside Police,-2.872892,53.488785,On or near Brook Hey Drive,E01006448,Knowsley 001A,Criminal damage and arson,Investigation complete; no suspect identified,NaN
3,489ca1cd895022d499d58ac9e627952d239292e33c8454...,2026-03,Merseyside Police,Merseyside Police,-2.870190,53.485658,On or near Darmond Road,E01006448,Knowsley 001A,Drugs,Under investigation,NaN
4,f9b6bf6a69cfa25e8e430c6d3403421bda8561db1b5b23...,2026-03,Merseyside Police,Merseyside Police,-2.874261,53.490168,On or near Kenbury Close,E01006448,Knowsley 001A,Other theft,Investigation complete; no suspect identified,NaN


## Cleaning & Validation

### Crime Severity

In [ ]:
# Check data types and overall size
sev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Crime Index     245 non-null    object
 1   Offence         250 non-null    object
 2   Weight          250 non-null    object
 3   Crime Category  242 non-null    object
dtypes: object(4)
memory usage: 7.9+ KB


***
**Data types should be:**  
Crime Index :   object  
Offense:        object  
Weight:         int64  
Crime Category: object  
  
**Conclusion:** Change weight to an integer
***

In [ ]:
# Check for nulls
sev.isnull().sum()

Crime Index       5
Offence           0
Weight            0
Crime Category    8
dtype: int64

***
**Crime Index has null values.**
- Looking at the data, it serves no purpose for the reason we need the datasheet for. It is worth dropping the column.  
**Conclusion**: Drop the column

***

**Crime Category has null values.**
- Crime category cannot be dropped, it is required.  
- Set crime category nulls to 'No Category', such that they are known as null.   
- Looking at the data, these offenses are related to cyber risks and hacking. These will not be found within the crime dataset, as it only logs in person activities.  
**Conclusion**: Set null values to 'No Category'
***

In [52]:
# Check for duplicates
print(sev.duplicated().sum())

3


***
**There are 3 duplicated rows in the dataset**  
No reason to keep the duplicated rows in the dataset, it is repeated data, and will skew averages.  
**Conclusion:** Remove dupllicated rows
***
***

In [49]:
# Change weight to an integer
sev['Weight'] = sev['Weight'].astype(str).str.replace(',', '', regex=False)
sev['Weight'] = pd.to_numeric(sev['Weight'], errors='coerce')

print(f'Weight datatype: {sev['Weight'].dtype}')
print(f'Number of nulls in Weight: {sev['Weight'].isnull().sum()}')
print(f'Sample of Weight:')
display(sev['Weight'].sample(3))

## 

Weight datatype: int64
Number of nulls in Weight: 0
Sample of Weight:


37     107
133     75
170    209
Name: Weight, dtype: int64

In [50]:
#Drop Crime Index
sev = sev.drop(columns=['Crime Index'])

print(f'Sample of severance weighting:')
display(sev.sample(3))

Sample of severance weighting:


,Offence,Weight,Crime Category
242,Department of Work and Pensions (DWP) fraud,63,Other crime
198,Dishonestly retaining a wrongful credit,208,Other crime
38,Child abduction,293,Violence and sexual offences


In [56]:
# Set null values of Crime Category to 'No Category'
sev['Crime Category'] = sev['Crime Category'].fillna('No Category')

print(sev['Crime Category'].value_counts())

Crime Category
Other crime                     91
Violence and sexual offences    83
Burglary                        16
Criminal damage and arson       12
Other theft                      8
Public order                     8
No Category                      8
Possession of weapons            7
Drugs                            5
Vehicle crime                    4
Robbery                          2
Theft from the person            1
Bicycle Theft                    1
Shoplifting                      1
Name: count, dtype: int64


In [57]:
# Remove duplicated rows
sev = sev.drop_duplicates()
print(sev.duplicated().sum())

0


***
***
Data in crime severity is now clean.

In [58]:
sev.info()

<class 'pandas.core.frame.DataFrame'>
Index: 247 entries, 0 to 249
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Offence         247 non-null    object
 1   Weight          247 non-null    int64 
 2   Crime Category  247 non-null    object
dtypes: int64(1), object(2)
memory usage: 7.7+ KB


In [59]:
sev.isnull().sum()

Offence           0
Weight            0
Crime Category    0
dtype: int64

### Crime Data

In [ ]:
# Looking at data
mssd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13356 entries, 0 to 13355
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               12058 non-null  object 
 1   Month                  13356 non-null  object 
 2   Reported by            13356 non-null  object 
 3   Falls within           13356 non-null  object 
 4   Longitude              13356 non-null  float64
 5   Latitude               13356 non-null  float64
 6   Location               13356 non-null  object 
 7   LSOA code              13356 non-null  object 
 8   LSOA name              13356 non-null  object 
 9   Crime type             13356 non-null  object 
 10  Last outcome category  12058 non-null  object 
 11  Context                0 non-null      float64
dtypes: float64(3), object(9)
memory usage: 1.2+ MB


In [21]:
mssd.isnull().sum()

Crime ID                  1298
Month                        0
Reported by                  0
Falls within                 0
Longitude                    0
Latitude                     0
Location                     0
LSOA code                    0
LSOA name                    0
Crime type                   0
Last outcome category     1298
Context                  13356
dtype: int64

Crime ID has null values - This means we need to create a new primary key for this database.
As each database is categorised by its location and its year and month, we will use that in its primary key.
eg: mers_2026_03_00001 - This allows for up to 100,000 crimes per month per police region.



## Feature Engineering & Transformation

#### Crime Severity

***
This section will create a processed dataset from the crime severity which will find the average weighting for each crime category, to be used in the aggregated dataset in order to aid in visualising where dangerous areas are, rather than where lots of crime happens. It will output in the /Data/Processed folder
***

In [ ]:
## Group Table
sev_by_crime_category = sev.groupby(['Crime Category'])

## Create Columns
num_items = sev_by_crime_category['Weight'].count()

mean_weight = sev_by_crime_category['Weight'].mean()

median_weight = sev_by_crime_category['Weight'].median()

min_weight = sev_by_crime_category['Weight'].min()

max_weight = sev_by_crime_category['Weight'].max()

std_deviation_weight = sev_by_crime_category['Weight'].std()

##Formulate Table
crime_category_severity = pd.DataFrame({
    'num_items': num_items,
    'mean_weight': mean_weight,
    'median_weight': median_weight,
    'min_weight': min_weight,
    'max_weight': max_weight,
    'std_deviation_weight': std_deviation_weight
})

## Visulaise Table
display(crime_category_severity)

,num_items,mean_weight,median_weight,min_weight,max_weight,std_deviation_weight
Crime Category,,,,,,
Bicycle Theft,1,16.000000,16.0,16,16,NaN
Burglary,16,703.250000,438.0,117,2127,697.764765
Criminal damage and arson,12,132.000000,19.0,7,837,255.916890
Drugs,5,105.000000,9.0,3,497,219.157478
No Category,8,280.250000,106.0,106,803,322.648305
Other crime,91,162.813187,86.0,4,4392,459.382603
Other theft,8,143.375000,51.5,7,803,268.795694
Possession of weapons,7,367.142857,75.0,55,1365,490.724101
Public order,8,405.500000,261.0,10,1880,615.286461


***
Looking at these stats, taking the median seems to give a better value, as the skew from large and small data is much less.  
Furthermore, by taking the average - given we are working with large datasets - the skew will become more obvious. This is because lower weighted crimes will be committed more often.  
With median: The more common crimes will be weighted slightly higher than they should, the more dangerous crimes will be rated much lower than they should.  
With mean: The more common crimes will be rated much higher than they should, the more dangerous crimes will be rated lower than they should.  
  
**Assumption:** Median is the best average to use for crime severity weighting.
***

In [ ]:
## Create Finalised Table
crime_category_severity_final = pd.DataFrame({
    'avg_weight': median_weight,
})

display(crime_category_severity_final) #remove to view table before exporting

## Validation Checks
print(crime_category_severity_final.info())

print(crime_category_severity_final.isnull().sum())

print(crime_category_severity_final.duplicated().sum())


## Output Final Table to csv
crime_category_severity_final.to_csv('../Data/Processed/crime-category-severity-weighting.csv', index=False)

print(f'File successfully created: {Path('../Data/Processed/crime-category-severity-weighting.csv').exists()}')

,avg_weight
Crime Category,
Bicycle Theft,16.0
Burglary,438.0
Criminal damage and arson,19.0
Drugs,9.0
No Category,106.0
Other crime,86.0
Other theft,51.5
Possession of weapons,75.0
Public order,261.0


<class 'pandas.core.frame.DataFrame'>
Index: 14 entries, Bicycle Theft to Violence and sexual offences
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   avg_weight  14 non-null     float64
dtypes: float64(1)
memory usage: 224.0+ bytes
None
avg_weight    0
dtype: int64
1
File successfully created: True


## Aggregation for Reporting

## Export